In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import  DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])


In [24]:
train_set = CIFAR10(root = "./CNN_Dataset" , train = True , download = True , transform = transform)
test_set = CIFAR10(root = "./CNN_Dataset" , train = False , download = True , transform = transform)


In [25]:
train_set

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./CNN_Dataset
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [26]:
test_set

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./CNN_Dataset
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [27]:
train_loader = DataLoader(train_set , shuffle = True , batch_size = 64 )
test_loader = DataLoader (test_set , batch_size = 64)

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x


In [29]:
model = CNN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [31]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)

        loss = criteria(outputs, labels)

        loss.backward()

        optimizer.step()

        epoch_training_loss += loss.item()

    print(
        f"Epoch = {epoch + 1}, "
        f"Loss = {epoch_training_loss / len(train_loader):.4f}"
    )


Epoch = 1, Loss = 1.3503
Epoch = 2, Loss = 0.9644
Epoch = 3, Loss = 0.8020
Epoch = 4, Loss = 0.6917
Epoch = 5, Loss = 0.6053
Epoch = 6, Loss = 0.5383
Epoch = 7, Loss = 0.4751
Epoch = 8, Loss = 0.4194
Epoch = 9, Loss = 0.3696
Epoch = 10, Loss = 0.3275


In [32]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = correct_labels / total_labels * 100

print(f"Accuracy = {accuracy:.2f}%")


Accuracy = 74.95%
